# BrainGNN notebook (Colab) on ABIDE

This is the notebook the pipeline was developed and run in (outputs cleared).

**Note:** it clones the *reference* repo and then patches it with `sed` / `%%writefile`. 
Those changes are already committed in this repository, so for a fresh run you can instead:

```
!git clone https://github.com/mohisha1205/BrainGNN_Pytorch.git
%cd BrainGNN_Pytorch
!pip install -r requirements.txt && python scripts/patch_deepdish.py
!python 01-fetch_data.py
!python imports/build_abide_dataset.py
!python 03-main.py --weightdecay 0.02 --ratio 0.5 --stepsize 10
```

In [ ]:
!git clone https://github.com/xxxhois/BrainGNN_Pytorch.git

In [ ]:
%cd BrainGNN_Pytorch

In [ ]:
# ===== CLEAN INSTALL FOR BRAINGNN =====

# !pip uninstall -y torch torch_geometric torch_scatter torch_sparse torch_cluster torch_spline_conv numpy deepdish

# numpy compatible with Python 3.12 + deepdish
# !pip install numpy==1.23.5

# core
!pip install pandas scikit-learn nilearn tensorboardX deepdish

# PyTorch (stable CPU version — safest)
# !pip install torch==2.1.0 torchvision==0.16.0

# PyG matching torch 2.1
!pip install torch_scatter torch_sparse torch_cluster torch_spline_conv torch_geometric
#  -f https://data.pyg.org/whl/torch-2.1.0+cpu.html

In [ ]:
# Install PyTorch Geometric extensions
# !pip install torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-1.13.0+cu117.html

# # Verify installation
# !python check_compatibility.py

In [ ]:
# !pip install nilearn tensorboardX

In [ ]:
!sed -i 's/from_numpy_matrix/from_numpy_array/g' /content/BrainGNN_Pytorch/imports/read_abide_stats_parall.py
!sed -i 's/nx.to_scipy_sparse_matrix/nx.to_scipy_sparse_array/g' /content/BrainGNN_Pytorch/imports/read_abide_stats_parall.py
!sed -i "/np.ComplexWarning/d" /usr/local/lib/python3.12/dist-packages/deepdish/core.py
!find /usr/local/lib/python3.12/dist-packages/deepdish/ -name "*.py" -exec sed -i "s/np.unicode_/np.str_/g" {} \;
!find /usr/local/lib/python3.12/dist-packages/deepdish/ -name "*.py" -exec sed -i "s/np.string_/np.bytes_/g" {} \;
!find /usr/local/lib/python3.12/dist-packages/deepdish/ -name "*.py" -exec sed -i "s/np.object/object/g" {} \;

In [ ]:
# !ls -l /content/BrainGNN_Pytorch/BrainGNN_Pytorch/imports/

In [ ]:
# !pip install torch-geometric deepdish

In [ ]:
# Download ABIDE dataset (50-200GB, may take hours)
!python 01-fetch_data.py



In [ ]:
import os
print(len(os.listdir('./data/ABIDE_pcp/cpac/filt_noglobal')))

In [ ]:
# Process data into graph format
!python 02-process_data.py

In [ ]:
import pandas as pd

pheno = pd.read_csv("./data/ABIDE_pcp/Phenotypic_V1_0b_preprocessed1.csv")

print(pheno.columns)
print(len(pheno))
pheno.head()

print(pheno["DX_GROUP"].value_counts())
print(pheno["SUB_ID"].head())

In [ ]:
import os
import pandas as pd

data_root = "./data/ABIDE_pcp/cpac/filt_noglobal"

# --- get only subject folders ---
subjects = [
    int(s) for s in os.listdir(data_root)
    if os.path.isdir(os.path.join(data_root, s)) and s.isdigit()
]

subjects = sorted(subjects)

print("Total subject folders:", len(subjects))
print("First 10 subjects:", subjects[:10])


# --- load phenotypic csv ---
pheno = pd.read_csv("./data/ABIDE_pcp/Phenotypic_V1_0b_preprocessed1.csv")

usable = pheno[pheno["SUB_ID"].isin(subjects)]

print("\nUsable subjects:", len(usable))
print("\nLabel distribution:")
print(usable["DX_GROUP"].value_counts())

In [ ]:
import os

root = './data/ABIDE_pcp/cpac/filt_noglobal'

for i,f in enumerate(os.listdir(root)):
    print(f)
    sub = os.listdir(os.path.join(root,f))
    print("   ", sub[:5])
    break

In [ ]:
# %cd imports/
# %pwd


In [ ]:
%%writefile /content/BrainGNN_Pytorch/imports/build_abide_dataset.py
import os
import numpy as np
import torch
from torch_geometric.data import Data
from scipy.io import loadmat
import pandas as pd
from tqdm import tqdm

from gdc import GDC


DATA_ROOT = "/content/BrainGNN_Pytorch/data/ABIDE_pcp/cpac/filt_noglobal"
PHENO_CSV = "/content/BrainGNN_Pytorch/data/ABIDE_pcp/Phenotypic_V1_0b_preprocessed1.csv"
SAVE_PATH = "/content/BrainGNN_Pytorch/data/processed/abide_graph_dataset.pt"

K = 10
THRESH = 0.05


def sparsify(A):
    np.fill_diagonal(A, 0)

    # remove weak correlations
    A[np.abs(A) < THRESH] = 0

    N = A.shape[0]
    mask = np.zeros_like(A)

    for i in range(N):
        idx = np.argsort(np.abs(A[i]))[::-1][:K]
        mask[i, idx] = 1

    A = A * mask
    A = np.maximum(A, A.T)

    return A


def normalize_adj(A):
    deg = np.sum(np.abs(A), axis=1)
    deg_inv_sqrt = 1.0 / np.sqrt(deg + 1e-6)
    return deg_inv_sqrt[:, None] * A * deg_inv_sqrt[None, :]


def build_dataset():

    pheno = pd.read_csv(PHENO_CSV)

    label_map = {
        row.SUB_ID: 0 if row.DX_GROUP == 2 else 1
        for _, row in pheno.iterrows()
    }

    subjects = [
        int(s) for s in os.listdir(DATA_ROOT)
        if os.path.isdir(os.path.join(DATA_ROOT, s)) and s.isdigit()
    ]
    subjects = sorted(subjects)

    # -------- PASS 1 : GLOBAL FEATURE NORMALIZATION --------

    all_features = []

    for sid in tqdm(subjects, desc="Collecting features"):
        corr_file = os.path.join(DATA_ROOT, str(sid), f"{sid}_cc200_correlation.mat")
        corr = loadmat(corr_file)["connectivity"]
        all_features.append(corr)

    all_features = np.vstack(all_features)

    global_mean = all_features.mean(axis=0, keepdims=True)
    global_std = all_features.std(axis=0, keepdims=True) + 1e-6

    print("Global normalization stats computed")

    # -------- GDC TRANSFORM (CONTROLLED SETTINGS) --------

    gdc = GDC(
        self_loop_weight=1,
        normalization_in='sym',
        normalization_out='col',
        diffusion_kwargs=dict(method='ppr', alpha=0.05),
        sparsification_kwargs=dict(method='threshold', avg_degree=8),
        exact=True
    )

    dataset = []
    edge_counts = []

    # -------- PASS 2 : BUILD GRAPHS --------

    for sid in tqdm(subjects, desc="Building graphs"):

        subj_dir = os.path.join(DATA_ROOT, str(sid))

        pcorr_file = os.path.join(subj_dir, f"{sid}_cc200_partial_correlation.mat")
        corr_file = os.path.join(subj_dir, f"{sid}_cc200_correlation.mat")

        pcorr = loadmat(pcorr_file)["connectivity"]
        corr = loadmat(corr_file)["connectivity"]

        pcorr = sparsify(pcorr)
        pcorr = normalize_adj(pcorr)

        row, col = np.nonzero(pcorr)

        edge_index = torch.tensor(np.stack([row, col]), dtype=torch.long)
        edge_attr = torch.tensor(pcorr[row, col], dtype=torch.float)

        X = (corr - global_mean) / global_std

        y = label_map[sid]

        pos = torch.eye(200)

        data = Data(
            x=torch.tensor(X, dtype=torch.float),
            edge_index=edge_index,
            edge_attr=edge_attr,
            y=torch.tensor([y], dtype=torch.long),
            pos=pos
        )

        # -------- APPLY CONTROLLED GDC --------
        data = gdc(data)

        dataset.append(data)
        edge_counts.append(data.edge_index.shape[1])

    print("\nDataset built.")
    print("Average edges per graph:", np.mean(edge_counts))

    os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)
    torch.save(dataset, SAVE_PATH)

    print("Saved dataset at:", SAVE_PATH)


if __name__ == "__main__":
    build_dataset()

In [ ]:
!python /content/BrainGNN_Pytorch/imports/build_abide_dataset.py

In [ ]:
import os

for root, dirs, files in os.walk(".", topdown=True):
    if "abide_graph_dataset.pt" in files:
        print("FOUND AT:", os.path.join(root, "abide_graph_dataset.pt"))

In [ ]:
# %cd ../

In [ ]:
import torch
file_path = '/content/BrainGNN_Pytorch/data/processed/abide_graph_dataset.pt'
dataset = torch.load(file_path, weights_only=False)
print(len(dataset))
print(dataset[0])

In [ ]:
# %%writefile /content/BrainGNN_Pytorch/03-main.py
# import os
# import copy
# import argparse
# import numpy as np
# import matplotlib.pyplot as plt

# import torch
# import torch.nn.functional as F
# from torch.optim import lr_scheduler

# from torch_geometric.loader import DataLoader
# from net.braingnn import Network
# from sklearn.model_selection import StratifiedKFold

# torch.manual_seed(123)

# EPS = 1e-10
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# parser = argparse.ArgumentParser()
# parser.add_argument('--n_epochs', type=int, default=150)
# parser.add_argument('--batchSize', type=int, default=16)
# parser.add_argument('--lr', type=float, default=0.001)
# parser.add_argument('--stepsize', type=int, default=20)
# parser.add_argument('--gamma', type=float, default=0.5)
# parser.add_argument('--weightdecay', type=float, default=0.02)

# parser.add_argument('--lamb0', type=float, default=1)
# parser.add_argument('--lamb1', type=float, default=0.1)
# parser.add_argument('--lamb2', type=float, default=0.1)
# parser.add_argument('--lamb3', type=float, default=0.1)
# parser.add_argument('--lamb4', type=float, default=0.1)
# parser.add_argument('--lamb5', type=float, default=0.1)

# parser.add_argument('--ratio', type=float, default=0.2)
# parser.add_argument('--indim', type=int, default=200)
# parser.add_argument('--nclass', type=int, default=2)

# parser.add_argument('--save_path', type=str, default='./cv_models/')
# opt = parser.parse_args()

# os.makedirs(opt.save_path, exist_ok=True)
# os.makedirs("./plots", exist_ok=True)

# ################ LOAD DATASET ################

# dataset = torch.load(
#     "./data/processed/abide_graph_dataset.pt",
#     weights_only=False
# )

# labels = np.array([d.y.item() for d in dataset])

# print("Dataset size:", len(dataset))


# ############ LOSSES ############

# def topk_loss(s, ratio):
#     if ratio > 0.5:
#         ratio = 1 - ratio
#     s = s.sort(dim=1).values
#     return -torch.log(s[:, -int(s.size(1) * ratio):] + EPS).mean() \
#            - torch.log(1 - s[:, :int(s.size(1) * ratio)] + EPS).mean()


# def consist_loss(s):
#     if len(s) == 0:
#         return 0
#     s = torch.sigmoid(s)
#     W = torch.ones(s.shape[0], s.shape[0]).to(device)
#     D = torch.eye(s.shape[0]).to(device) * torch.sum(W, dim=1)
#     L = D - W
#     return torch.trace(s.t() @ L @ s) / (s.shape[0] * s.shape[0])


# ############ TRAIN / TEST FUNCTIONS ############

# def train_epoch(model, loader, optimizer, scheduler):
#     model.train()
#     loss_all = 0

#     for data in loader:
#         data = data.to(device)
#         optimizer.zero_grad()

#         output, w1, w2, s1, s2 = model(
#             data.x, data.edge_index, data.batch, data.edge_attr, data.pos
#         )

#         loss_c = F.nll_loss(output, data.y)

#         loss_p1 = (torch.norm(w1, p=2) - 1) ** 2
#         loss_p2 = (torch.norm(w2, p=2) - 1) ** 2

#         loss_tpk1 = topk_loss(s1, opt.ratio)
#         loss_tpk2 = topk_loss(s2, opt.ratio)

#         loss_consist = 0
#         for c in range(opt.nclass):
#             loss_consist += consist_loss(s1[data.y == c])

#         loss = (
#             opt.lamb0 * loss_c
#             + opt.lamb1 * loss_p1
#             + opt.lamb2 * loss_p2
#             + opt.lamb3 * loss_tpk1
#             + opt.lamb4 * loss_tpk2
#             + opt.lamb5 * loss_consist
#         )

#         loss.backward()
#         optimizer.step()
#         loss_all += loss.item() * data.num_graphs

#     scheduler.step()
#     return loss_all / len(loader.dataset)


# def eval_acc(model, loader):
#     model.eval()
#     correct = 0
#     with torch.no_grad():
#         for data in loader:
#             data = data.to(device)
#             output = model(
#                 data.x, data.edge_index, data.batch, data.edge_attr, data.pos
#             )[0]
#             pred = output.max(1)[1]
#             correct += pred.eq(data.y).sum().item()
#     return correct / len(loader.dataset)


# ############ STRATIFIED 5-FOLD CV ############

# skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# fold_results = []

# for fold, (train_val_idx, test_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):

#     print(f"\n========== FOLD {fold+1} ==========")

#     train_idx = train_val_idx[:int(0.8 * len(train_val_idx))]
#     val_idx = train_val_idx[int(0.8 * len(train_val_idx)):]

#     train_loader = DataLoader([dataset[i] for i in train_idx], batch_size=opt.batchSize, shuffle=True)
#     val_loader   = DataLoader([dataset[i] for i in val_idx], batch_size=opt.batchSize)
#     test_loader  = DataLoader([dataset[i] for i in test_idx], batch_size=opt.batchSize)

#     model = Network(opt.indim, opt.ratio, opt.nclass).to(device)

#     optimizer = torch.optim.Adam(model.parameters(), lr=opt.lr, weight_decay=opt.weightdecay)
#     scheduler = lr_scheduler.StepLR(optimizer, step_size=opt.stepsize, gamma=opt.gamma)

#     best_val = 0
#     patience = 25
#     counter = 0
#     best_wts = copy.deepcopy(model.state_dict())

#     train_acc_hist = []
#     val_acc_hist = []

#     for epoch in range(opt.n_epochs):

#         loss = train_epoch(model, train_loader, optimizer, scheduler)

#         tr_acc = eval_acc(model, train_loader)
#         val_acc = eval_acc(model, val_loader)

#         train_acc_hist.append(tr_acc)
#         val_acc_hist.append(val_acc)

#         print(f"Epoch {epoch:03d} | Loss {loss:.4f} | Train {tr_acc:.4f} | Val {val_acc:.4f}")

#         if val_acc > best_val:
#             best_val = val_acc
#             counter = 0
#             best_wts = copy.deepcopy(model.state_dict())
#             torch.save(best_wts, f"{opt.save_path}/fold{fold+1}.pth")
#         else:
#             counter += 1

#         if counter >= patience:
#             print("Early stopping")
#             break

#     model.load_state_dict(best_wts)
#     test_acc = eval_acc(model, test_loader)

#     print("Fold Test Accuracy:", test_acc)
#     fold_results.append(test_acc)

#     plt.figure()
#     plt.plot(train_acc_hist, label="Train")
#     plt.plot(val_acc_hist, label="Val")
#     plt.title(f"Fold {fold+1}")
#     plt.legend()
#     plt.savefig(f"./plots/fold{fold+1}.png")


# print("\n================ FINAL RESULTS ================")
# print("Fold Accuracies:", fold_results)
# print("Mean Accuracy:", np.mean(fold_results))
# print("Std:", np.std(fold_results))

In [ ]:
%%writefile /content/BrainGNN_Pytorch/03-main.py
import os
import copy
import argparse
import csv
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
from torch.optim import lr_scheduler

from torch_geometric.loader import DataLoader
from net.braingnn import Network
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import confusion_matrix

torch.manual_seed(123)
np.random.seed(123)

EPS = 1e-10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

parser = argparse.ArgumentParser()
parser.add_argument('--n_epochs', type=int, default=150)
parser.add_argument('--batchSize', type=int, default=16)
parser.add_argument('--lr', type=float, default=0.001)
parser.add_argument('--stepsize', type=int, default=20)
parser.add_argument('--gamma', type=float, default=0.5)
parser.add_argument('--weightdecay', type=float, default=0.02)

parser.add_argument('--lamb0', type=float, default=1)
parser.add_argument('--lamb1', type=float, default=0.1)
parser.add_argument('--lamb2', type=float, default=0.1)
parser.add_argument('--lamb3', type=float, default=0.1)
parser.add_argument('--lamb4', type=float, default=0.1)
parser.add_argument('--lamb5', type=float, default=0.1)

parser.add_argument('--ratio', type=float, default=0.2)
parser.add_argument('--indim', type=int, default=200)
parser.add_argument('--nclass', type=int, default=2)

parser.add_argument('--save_path', type=str, default='./cv_models/')
parser.add_argument('--plot_path', type=str, default='./plots/')
parser.add_argument('--csv_path', type=str, default='./plots/fold_metrics.csv')
parser.add_argument('--val_size', type=float, default=0.2)

opt = parser.parse_args()

os.makedirs(opt.save_path, exist_ok=True)
os.makedirs(opt.plot_path, exist_ok=True)

dataset = torch.load("./data/processed/abide_graph_dataset.pt", weights_only=False)
labels = np.array([d.y.item() for d in dataset])

print("Dataset size:", len(dataset))

def topk_loss(s, ratio):
    if ratio > 0.5:
        ratio = 1 - ratio
    s = s.sort(dim=1).values
    k = max(1, int(s.size(1) * ratio))
    return -torch.log(s[:, -k:] + EPS).mean() - torch.log(1 - s[:, :k] + EPS).mean()

def consist_loss(s):
    if len(s) == 0:
        return 0
    s = torch.sigmoid(s)
    W = torch.ones(s.shape[0], s.shape[0]).to(device)
    D = torch.eye(s.shape[0]).to(device) * torch.sum(W, dim=1)
    L = D - W
    return torch.trace(s.t() @ L @ s) / (s.shape[0] * s.shape[0])

def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    loss_all = 0.0

    for data in loader:
        data = data.to(device)
        optimizer.zero_grad()

        output, w1, w2, s1, s2 = model(
            data.x, data.edge_index, data.batch, data.edge_attr, data.pos
        )

        loss_c = F.nll_loss(output, data.y)
        loss_p1 = (torch.norm(w1, p=2) - 1) ** 2
        loss_p2 = (torch.norm(w2, p=2) - 1) ** 2
        loss_tpk1 = topk_loss(s1, opt.ratio)
        loss_tpk2 = topk_loss(s2, opt.ratio)

        loss_consist = 0
        for c in range(opt.nclass):
            cls_mask = (data.y == c)
            if cls_mask.sum() > 0:
                loss_consist += consist_loss(s1[cls_mask])

        loss = (
            opt.lamb0 * loss_c
            + opt.lamb1 * loss_p1
            + opt.lamb2 * loss_p2
            + opt.lamb3 * loss_tpk1
            + opt.lamb4 * loss_tpk2
            + opt.lamb5 * loss_consist
        )

        loss.backward()
        optimizer.step()
        loss_all += loss.item() * data.num_graphs

    scheduler.step()
    return loss_all / len(loader.dataset)

def eval_metrics(model, loader):
    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            output = model(data.x, data.edge_index, data.batch, data.edge_attr, data.pos)[0]
            pred = output.max(1)[1]
            y_true.extend(data.y.cpu().numpy().tolist())
            y_pred.extend(pred.cpu().numpy().tolist())

    cm = confusion_matrix(y_true, y_pred, labels=[0,1])

    if cm.shape != (2, 2):
        tn = fp = fn = tp = 0
        if len(y_true) > 0:
            unique = sorted(set(y_true + y_pred))
            if unique == [0]:
                tn = len(y_true)
            elif unique == [1]:
                tp = len(y_true)
    else:
        tn, fp, fn, tp = cm.ravel()

    acc = (tp + tn) / (tp + tn + fp + fn + EPS)
    sensitivity = tp / (tp + fn + EPS)
    specificity = tn / (tn + fp + EPS)

    return acc, sensitivity, specificity

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_results = []

for fold, (train_val_idx, test_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
    print(f"\n========== FOLD {fold + 1} ==========")

    train_val_labels = labels[train_val_idx]

    inner_split = StratifiedShuffleSplit(
        n_splits=1,
        test_size=opt.val_size,
        random_state=42 + fold
    )

    inner_train_rel, inner_val_rel = next(inner_split.split(np.zeros(len(train_val_idx)), train_val_labels))
    train_idx = train_val_idx[inner_train_rel]
    val_idx = train_val_idx[inner_val_rel]

    train_loader = DataLoader([dataset[i] for i in train_idx], batch_size=opt.batchSize, shuffle=True)
    val_loader = DataLoader([dataset[i] for i in val_idx], batch_size=opt.batchSize, shuffle=False)
    test_loader = DataLoader([dataset[i] for i in test_idx], batch_size=opt.batchSize, shuffle=False)

    model = Network(opt.indim, opt.ratio, opt.nclass).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=opt.lr, weight_decay=opt.weightdecay)
    scheduler = lr_scheduler.StepLR(optimizer, step_size=opt.stepsize, gamma=opt.gamma)

    best_val = -1.0
    patience = 12
    counter = 0
    best_wts = copy.deepcopy(model.state_dict())

    train_acc_hist = []
    val_acc_hist = []

    for epoch in range(opt.n_epochs):
        loss = train_epoch(model, train_loader, optimizer, scheduler)
        tr_acc, _, _ = eval_metrics(model, train_loader)
        val_acc, _, _ = eval_metrics(model, val_loader)

        train_acc_hist.append(tr_acc)
        val_acc_hist.append(val_acc)

        print(f"Epoch {epoch:03d} | Loss {loss:.4f} | Train {tr_acc:.4f} | Val {val_acc:.4f}")

        if val_acc > best_val:
            best_val = val_acc
            counter = 0
            best_wts = copy.deepcopy(model.state_dict())
            torch.save(best_wts, f"{opt.save_path}/fold{fold + 1}.pth")
        else:
            counter += 1

        if counter >= patience:
            print("Early stopping")
            break

    model.load_state_dict(best_wts)
    test_acc, test_sens, test_spec = eval_metrics(model, test_loader)

    print("Fold Test Accuracy:", test_acc)
    print("Fold Sensitivity:", test_sens)
    print("Fold Specificity:", test_spec)

    fold_results.append({
        "fold": fold + 1,
        "accuracy": test_acc,
        "sensitivity": test_sens,
        "specificity": test_spec
    })

    plt.figure()
    plt.plot(train_acc_hist, label="Train")
    plt.plot(val_acc_hist, label="Val")
    plt.title(f"Fold {fold + 1}")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{opt.plot_path}/fold{fold + 1}.png")
    plt.close()

with open(opt.csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["fold", "accuracy", "sensitivity", "specificity"])
    writer.writeheader()
    writer.writerows(fold_results)

accs = [x["accuracy"] for x in fold_results]
sens = [x["sensitivity"] for x in fold_results]
spec = [x["specificity"] for x in fold_results]

print("\n================ FINAL RESULTS ================")
print("Fold Accuracies:", accs)
print("Mean Accuracy:", np.mean(accs))
print("Std Accuracy:", np.std(accs))
print("Mean Sensitivity:", np.mean(sens))
print("Std Sensitivity:", np.std(sens))
print("Mean Specificity:", np.mean(spec))
print("Std Specificity:", np.std(spec))
print("Saved CSV to:", opt.csv_path)

In [ ]:
!python 03-main.py \
--weightdecay 0.02 \
--ratio 0.5 \
--stepsize 10
  # --batchSize 32 \
  # --n_epochs 100 \
  # --lr 0.001 \
  # --stepsize 20 \
  # --gamma 0.5 \



In [ ]:
# !find /content -type f -name "training_curves.png"

In [ ]:
# from IPython.display import Image

# Image('/content/BrainGNN_Pytorch/training_curves.png')


In [ ]:
# %load_ext tensorboard
# %tensorboard --logdir /content/BrainGNN_Pytorch/log

In [ ]:
# !tensorboard --logdir=./log

In [ ]:
# !zip -r /content/content_files.zip /content/


In [ ]:
# from google.colab import files
# files.download("/content/content_files.zip")


In [ ]:
import os

# Create the directory if it doesn't exist
output_dir = '/content/BrainGNN_Pytorch/plots'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Zip the plots folder
!zip -r /content/brain_gnn_plots.zip {output_dir}

In [ ]:
from google.colab import files
files.download('/content/brain_gnn_plots.zip')

In [ ]:
import scipy.io
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Load the .mat file (Replace with your actual file path)
file_path = '/content/BrainGNN_Pytorch/data/ABIDE_pcp/cpac/filt_noglobal/50003/50003_cc200_correlation.mat'
mat_data = scipy.io.loadmat(file_path)

# 2. .mat files have "keys" (like a dictionary). Let's see what they are:
print("Variables inside this .mat file:", mat_data.keys())

# 3. Extract the actual matrix.
# Look at the print statement above. The matrix is usually stored under a key
# like 'connectivity', 'corr_matrix', 'ROI_data', or just the name of the atlas.
# Replace 'YOUR_KEY_HERE' with the correct key from step 2.
brain_matrix = mat_data['connectivity']

# 4. Plot it as a heatmap!
plt.figure(figsize=(10, 8))
# cmap='coolwarm' makes strong negative correlations BLUE and strong positive RED
sns.heatmap(brain_matrix, cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title(f"Correlation Matrix: {file_path}")
plt.xlabel("Brain Region (ROI)")
plt.ylabel("Brain Region (ROI)")
plt.show()

In [ ]:
import scipy.io
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Load the .mat file (Replace with your actual file path)
file_path = '/content/BrainGNN_Pytorch/data/ABIDE_pcp/cpac/filt_noglobal/50003/50003_cc200_partial_correlation.mat'
mat_data = scipy.io.loadmat(file_path)

# 2. .mat files have "keys" (like a dictionary). Let's see what they are:
print("Variables inside this .mat file:", mat_data.keys())

# 3. Extract the actual matrix.
# Look at the print statement above. The matrix is usually stored under a key
# like 'connectivity', 'corr_matrix', 'ROI_data', or just the name of the atlas.
# Replace 'YOUR_KEY_HERE' with the correct key from step 2.
brain_matrix = mat_data['connectivity']

# 4. Plot it as a heatmap!
plt.figure(figsize=(10, 8))
# cmap='coolwarm' makes strong negative correlations BLUE and strong positive RED
sns.heatmap(brain_matrix, cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title(f"Partial Correlation Matrix: {file_path}")
plt.xlabel("Brain Region (ROI)")
plt.ylabel("Brain Region (ROI)")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load the text file using NumPy
file_path = '/content/BrainGNN_Pytorch/data/ABIDE_pcp/cpac/filt_noglobal/50003/Pitt_0050003_rois_cc200.1D'
time_series = np.loadtxt(file_path)

print(f"Data shape: {time_series.shape}")
print("(Rows = Timepoints, Columns = Brain Regions)")

# -- VISUALIZATION A: Plot the BOLD signals over time --
plt.figure(figsize=(12, 4))
# Let's just plot the brain waves for the first 3 regions so it's not too messy
plt.plot(time_series[:, :3])
plt.title("BOLD Signal Over Time (First 3 Brain Regions)")
plt.xlabel("Time (Scans)")
plt.ylabel("Oxygen Level (Signal)")
plt.show()

# -- VISUALIZATION B: Turn the Time Series into a Connectivity Matrix --
# If you want to calculate the Pearson Correlation yourself!
# We transpose (.T) so regions are rows and timepoints are columns
corr_matrix = np.corrcoef(time_series.T)

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title("Generated Correlation Matrix from .1D Data")
plt.show()